In [27]:
from modules.preprocessing import *
from modules.modeling import *
from modules.visualizing import *

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [11]:

input_path = "data/"
output_path = "output/"

input_file_name = "data_market_tsne.csv"
df = pd.read_csv(os.path.join(input_path, input_file_name))

col_smiles = "SMILES"
col_index = "INCHIKEY"

In [ ]:
# Standardize SMILES
df['standardized SMILES'] = standardize_smiles_df(df, col_smiles)
# df['standardized tautomer SMILES'] = standardize_smiles_df(df, 'SMILES', tautomerize=True)

In [ ]:
# Calculate Morgan fingerprints
fingerprints = calculate_descriptors_morgan_df(df, 'standardized SMILES', radius=2, fpSize=1024)
df_fingerprints = pd.concat([df, fingerprints], axis=1)
df_fingerprints.to_csv(os.path.join(input_path, "df_fingerprints.csv"), index=False)


In [9]:
# Fit TSNE model
hyperparameters = dict(n_components=2, perplexity=100, n_iter=2000, learning_rate='auto', neighbors='pynndescent', initialization='pca', metric='jaccard', random_state=42, verbose=True)
# tsne, df_tsne = fit_tsne_model(df_fingerprints, hyperparameters)
df_fingerprints = pd.read_csv(os.path.join(input_path, "df_fingerprints.csv"))

In [12]:
df_tsne.to_csv(os.path.join(output_path, "df_tsne_sklearn.csv"), index=True)

In [ ]:
from openTSNE.sklearn import TSNE

# df_fingerprints_train = df_fingerprints[~(df_fingerprints['standardized SMILES']=='')]

fps = np.array(df_fingerprints.iloc[:, -1024:].astype('bool'))

tsne = TSNE(**hyperparameters)
df_tsne = pd.DataFrame(tsne.fit_transform(fps), columns=['TSNE1', 'TSNE2'])
df_tsne.index = df_fingerprints[col_index]
df_tsne.to_csv(os.path.join(output_path, "df_tsne_sklearn.csv"), index=True)

In [13]:
from openTSNE.tsne import TSNE

fps = np.array(df_fingerprints.iloc[:, -1024:].astype('bool'))

tsne = TSNE(**hyperparameters)
tsne = tsne.fit(fps)
df_tsne_2 = pd.DataFrame(tsne.transform(fps), columns = ['TSNE1', 'TSNE2'])
df_tsne_2.index = df_fingerprints[col_index]
df_tsne_2.to_csv(os.path.join(output_path, "df_tsne_opentsne.csv"), index=True)

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, metric='jaccard', n_iter=2000,
     neighbors='pynndescent', perplexity=100, random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 300 nearest neighbors using NN descent approximate search using jaccard distance...
   --> Time elapsed: 197.31 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 11.76 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 1.60 seconds
===> Running optimization with exaggeration=12.00, lr=11177.42 for 250 iterations...
Iteration   50, KL divergence 6.5451, 50 iterations in 20.3287 sec
Iteration  100, KL divergence 6.5703, 50 iterations in 21.9356 sec
Iteration  150, KL divergence 6.5755, 50 iterations in 22.2758 sec
Iteration  200, KL divergence 6.5778, 50 iterations in 22.9227 sec
Iteration  250, KL divergence 6.5781, 50 iterations in 21.1033 sec

In [ ]:
df_tsne = pd.read_csv(os.path.join(output_path, "df_tsne_sklearn.csv"))
df_tsne_2 = pd.read_csv(os.path.join(output_path, "df_tsne_opentsne.csv"))

df_tsne_sklearn = pd.merge(df.drop(columns=['TSNE1', 'TSNE2']), df_tsne, on=col_index)
df_tsne_opentsne = pd.merge(df.drop(columns=['TSNE1', 'TSNE2']), df_tsne_2, on=col_index) 

In [25]:
import plotly.graph_objects as go

In [28]:
# Visualize marketed chemical space (sklearn)
top15 = df_tsne_sklearn.groupby('Superclass').count()['TSNE1'].sort_values(ascending=False).index[:15]
df_tsne_sklearn['Superclass (top 15)'] = df_tsne_sklearn['Superclass'].where(df_tsne_sklearn['Superclass'].isin(top15), 'Other')

hue_order = top15.sort_values().to_list() + ['Other']
palette = ['darkslategrey', 'teal', 'aquamarine', 'darkred', 'orangered', 'mediumpurple', 'darkorchid',
            'mediumblue', 'royalblue', 'skyblue', 'darkgoldenrod', 'darkorange', 'gold',  'lightpink', 'hotpink',
            'lightgrey']

palette_hex = [mcolors.to_hex(c) for c in palette]
color_map = dict(zip(hue_order, palette_hex)) 

fig = chemical_space_plot(df_tsne_sklearn, 'Superclass (top 15)', color_map)
fig.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    font_color='black'
)
fig.show()

fig.write_html('../output/img_chemical_space_sklarn.html')

NameError: name 'go' is not defined

In [ ]:
# Test standardization with single SMILEs
smiles = 'C[C@H](O)[C@@H](C)O'
standardize_smiles(smiles, tautomerize=True)